# 🫁 Pneumonia Detection from Chest X-Ray

**Medical Image Processing & Deep Learning Project**

This notebook implements a reproducible research pipeline for binary classification of chest X-ray images into:

- `NORMAL`
- `PNEUMONIA`

The project compares three architectures:

1. **ConvNeXt V2-style Baseline**
2. **Pure Vision Transformer (ViT)**
3. **Hybrid ConvNeXt V2-style + Multi-Head Attention (proposed)**

The image pipeline also applies **CLAHE** preprocessing and data augmentation. Model evaluation includes accuracy, recall, classification report, confusion matrix, ROC-AUC, and an optional Grad-CAM-style explainability section.

> ⚠️ **Research / educational use only.** This project is not a medical diagnostic device and must not be used as a substitute for professional clinical assessment.

## 1. Environment

The original experiments were performed in **Google Colab** with Python/TensorFlow and GPU acceleration.

The notebook is intentionally self-contained so that it can be opened directly from GitHub in Google Colab.

In [ ]:
# Install the main dependencies in Google Colab
!pip -q install kaggle gradio opencv-python-headless seaborn scikit-learn

import os
import random
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Dataset

The project uses the public **Chest X-Ray Images (Pneumonia)** dataset by Paul Mooney on Kaggle.

The notebook does **not** include the dataset itself. Downloading the dataset at runtime keeps the GitHub repository lightweight and avoids committing large medical-image files.

### Expected directory structure

```text
dataset/
└── chest_xray/
    ├── train/
    │   ├── NORMAL/
    │   └── PNEUMONIA/
    ├── val/
    │   ├── NORMAL/
    │   └── PNEUMONIA/
    └── test/
        ├── NORMAL/
        └── PNEUMONIA/
```

In [ ]:
# Kaggle authentication
# In Colab, upload your kaggle.json file when prompted.
from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Please upload kaggle.json from your Kaggle account.")

    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(uploaded["kaggle.json"])

    os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Download and extract the dataset
!rm -rf dataset chest-xray-pneumonia.zip
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d dataset/

DATA_ROOT = "/content/dataset/chest_xray"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
TEST_DIR = os.path.join(DATA_ROOT, "test")

print("Dataset root:", DATA_ROOT)
print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists:", os.path.exists(VAL_DIR))
print("Test exists:", os.path.exists(TEST_DIR))

## 3. CLAHE Preprocessing and Data Augmentation

The preprocessing pipeline follows the project implementation:

1. Convert the image to `uint8`.
2. Convert RGB to grayscale.
3. Apply **CLAHE** with `clipLimit=2.0` and `tileGridSize=(8, 8)`.
4. Convert the enhanced grayscale image back to 3-channel RGB.
5. Normalize pixel values using `rescale=1./255`.

Training augmentation:

- Rotation: `10°`
- Zoom: `0.1`
- Width shift: `0.1`
- Height shift: `0.1`
- Horizontal flip: enabled
- Fill mode: `nearest`

All models use an input size of **224 × 224 × 3**.

In [ ]:
def pneumonia_preprocessing(img):
    img = img.astype(np.uint8)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(gray)

    final_img = cv2.cvtColor(
        enhanced,
        cv2.COLOR_GRAY2RGB
    )

    return final_img.astype(np.float32)


train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    preprocessing_function=pneumonia_preprocessing,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode="nearest",
)

val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    preprocessing_function=pneumonia_preprocessing,
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

print("Class indices:", train_generator.class_indices)

In [ ]:
# Visualize the effect of preprocessing
sample_batch, sample_labels = next(test_generator)

plt.figure(figsize=(12, 6))
for i in range(min(6, len(sample_batch))):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_batch[i])
    label = "PNEUMONIA" if sample_labels[i] == 1 else "NORMAL"
    plt.title(label)
    plt.axis("off")

plt.suptitle("Preprocessed Chest X-Ray Samples")
plt.tight_layout()
plt.show()

## 4. Shared ConvNeXt V2-style Components

The project implements a lightweight ConvNeXt V2-style architecture directly in Keras.

The implementation contains:

- Depthwise convolution
- Layer normalization
- Pointwise expansion
- GELU activation
- Global Response Normalization (GRN)
- Pointwise projection
- Residual connection

In [ ]:
class GRN(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        channels = input_shape[-1]
        self.gamma = self.add_weight(
            shape=(1, 1, 1, channels),
            initializer="zeros",
            trainable=True,
            name="gamma",
        )
        self.beta = self.add_weight(
            shape=(1, 1, 1, channels),
            initializer="zeros",
            trainable=True,
            name="beta",
        )

    def call(self, x):
        gx = tf.norm(
            x,
            ord="euclidean",
            axis=(1, 2),
            keepdims=True,
        )
        nx = gx / (
            tf.reduce_mean(
                gx,
                axis=-1,
                keepdims=True,
            ) + 1e-6
        )
        return self.gamma * (x * nx) + self.beta + x


def convnext_v2_block(x, dim):
    shortcut = x

    x = layers.DepthwiseConv2D(
        kernel_size=7,
        padding="same",
    )(x)
    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)
    x = layers.Conv2D(
        4 * dim,
        kernel_size=1,
    )(x)
    x = layers.Activation("gelu")(x)
    x = GRN()(x)
    x = layers.Conv2D(
        dim,
        kernel_size=1,
    )(x)

    return layers.Add()([shortcut, x])

## 5. Model A — ConvNeXt V2-style Baseline

In [ ]:
def build_convnext_v2_baseline(
    input_shape=(224, 224, 3)
):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    # Stage 1
    x = layers.Conv2D(
        96, kernel_size=4, strides=4, padding="same"
    )(x)
    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    for _ in range(2):
        x = convnext_v2_block(x, 96)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    # Stage 2
    x = layers.Conv2D(
        192, kernel_size=2, strides=2, padding="same"
    )(x)

    for _ in range(2):
        x = convnext_v2_block(x, 192)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    # Stage 3
    x = layers.Conv2D(
        384, kernel_size=2, strides=2, padding="same"
    )(x)

    for _ in range(4):
        x = convnext_v2_block(x, 384)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    return models.Model(
        inputs,
        outputs,
        name="ConvNeXt_V2_Baseline",
    )


model_baseline = build_convnext_v2_baseline()

model_baseline.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-4,
        weight_decay=1e-4,
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model_baseline.summary()

## 6. Model B — Pure Vision Transformer

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[
                1,
                self.patch_size,
                self.patch_size,
                1,
            ],
            strides=[
                1,
                self.patch_size,
                self.patch_size,
                1,
            ],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )

        patch_dims = patches.shape[-1]
        patches = tf.reshape(
            patches,
            [batch_size, -1, patch_dims],
        )

        return patches


class PatchEncoder(layers.Layer):
    def __init__(
        self,
        num_patches,
        projection_dim,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection = layers.Dense(
            units=projection_dim
        )
        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim,
        )

    def call(self, patch):
        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1,
        )
        encoded = (
            self.projection(patch)
            + self.position_embedding(positions)
        )
        return encoded


def build_pure_vit(
    input_shape=(224, 224, 3),
    patch_size=16,
    projection_dim=128,
    num_heads=4,
    transformer_layers=4,
):
    inputs = layers.Input(shape=input_shape)

    num_patches = (
        input_shape[0] // patch_size
    ) ** 2

    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(
        num_patches,
        projection_dim,
    )(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(
            epsilon=1e-6
        )(encoded_patches)

        attention_output = (
            layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=projection_dim,
                dropout=0.1,
            )(x1, x1)
        )

        x2 = layers.Add()([
            attention_output,
            encoded_patches,
        ])

        x3 = layers.LayerNormalization(
            epsilon=1e-6
        )(x2)
        x3 = layers.Dense(
            projection_dim * 2,
            activation=tf.nn.gelu,
        )(x3)
        x3 = layers.Dropout(0.1)(x3)
        x3 = layers.Dense(
            projection_dim
        )(x3)
        x3 = layers.Dropout(0.1)(x3)

        encoded_patches = layers.Add()([
            x3,
            x2,
        ])

    representation = layers.LayerNormalization(
        epsilon=1e-6
    )(encoded_patches)
    representation = layers.GlobalAveragePooling1D()(
        representation
    )
    representation = layers.Dropout(0.5)(
        representation
    )

    outputs = layers.Dense(
        1,
        activation="sigmoid",
    )(representation)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="Pure_ViT",
    )


model_pure_vit = build_pure_vit()

model_pure_vit.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-4,
        weight_decay=1e-4,
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model_pure_vit.summary()

## 7. Model C — Hybrid ConvNeXt V2-style + Multi-Head Attention

The proposed hybrid architecture first extracts local visual features with ConvNeXt V2-style blocks and then reshapes the feature map into a sequence.

Multi-Head Self-Attention is then used to model relationships between spatial feature tokens before classification.

In [ ]:
def build_convnext_vit_hybrid(
    input_shape=(224, 224, 3)
):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    # ConvNeXt V2-style feature extraction
    x = layers.Conv2D(
        96, kernel_size=4, strides=4, padding="same"
    )(x)
    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    for _ in range(2):
        x = convnext_v2_block(x, 96)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)
    x = layers.Conv2D(
        192, kernel_size=2, strides=2, padding="same"
    )(x)

    for _ in range(2):
        x = convnext_v2_block(x, 192)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)
    x = layers.Conv2D(
        384, kernel_size=2, strides=2, padding="same"
    )(x)

    for _ in range(4):
        x = convnext_v2_block(x, 384)

    # Convert spatial feature map to tokens
    seq = layers.Reshape((-1, 384))(x)

    attention_out = layers.MultiHeadAttention(
        num_heads=8,
        key_dim=64,
        dropout=0.3,
    )(seq, seq)

    seq = layers.Add()([
        attention_out,
        seq,
    ])

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(seq)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    return models.Model(
        inputs,
        outputs,
        name="ConvNeXtV2_ViT_Hybrid",
    )


model_hybrid = build_convnext_vit_hybrid()

model_hybrid.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-4,
        weight_decay=1e-4,
    ),
    loss=tf.keras.losses.BinaryCrossentropy(
        label_smoothing=0.1
    ),
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model_hybrid.summary()

## 8. Training

The original project used:

- Baseline: 15 epochs
- Pure ViT: 15 epochs
- Hybrid: up to 20 epochs
- AdamW learning rate: `1e-4`
- Weight decay: `1e-4`
- Hybrid label smoothing: `0.1`
- Hybrid class weighting: `{0: 2.8, 1: 1.0}`
- ReduceLROnPlateau
- EarlyStopping
- ModelCheckpoint

Training is intentionally placed in separate cells so that a user can train only the model they need.

In [ ]:
EPOCHS_BASELINE = 15

history_baseline = model_baseline.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_BASELINE,
)

In [ ]:
EPOCHS_VIT = 15

history_pure_vit = model_pure_vit.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_VIT,
)

In [ ]:
class_weight = {
    0: 2.8,
    1: 1.0,
}

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1,
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

checkpoint = ModelCheckpoint(
    "best_hybrid_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
)

EPOCHS_HYBRID = 20

history_hybrid = model_hybrid.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HYBRID,
    class_weight=class_weight,
    callbacks=[
        reduce_lr,
        early_stop,
        checkpoint,
    ],
)

## 9. Save Models

Model files are deliberately excluded from the normal Git history because they can be large. For GitHub, use Git LFS, GitHub Releases, or an external model-hosting service if you want to distribute weights.

In [ ]:
# Optional: save trained models locally in the Colab runtime.
model_baseline.save("convnext_v2_baseline.keras")
model_pure_vit.save("pure_vit.keras")
model_hybrid.save("best_hybrid_model.keras")

print("Models saved in the current Colab runtime.")

## 10. Evaluation Utilities

In [ ]:
def evaluate_binary_model(
    model,
    generator,
    model_name,
    threshold=0.5,
):
    generator.reset()

    scores = model.predict(
        generator,
        verbose=1,
    ).ravel()

    y_true = generator.classes
    y_pred = (
        scores >= threshold
    ).astype(int)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_true, y_pred
        ),
        "Precision": precision_score(
            y_true, y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred,
            zero_division=0
        ),
        "AUC": roc_auc_score(
            y_true, scores
        ),
    }

    print(f"\n===== {model_name} =====")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Normal",
                "Pneumonia",
            ],
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
    )

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[
            "Normal",
            "Pneumonia",
        ],
        yticklabels=[
            "Normal",
            "Pneumonia",
        ],
    )
    plt.title(
        f"Confusion Matrix - {model_name}"
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    return metrics, y_true, scores

## 11. Evaluate All Three Models

The default threshold is `0.5`. If the experiment uses a different threshold, set it explicitly and report it together with the metrics.

In [ ]:
results = []

baseline_metrics, y_true, baseline_scores = evaluate_binary_model(
    model_baseline,
    test_generator,
    "ConvNeXt V2 Baseline",
    threshold=0.5,
)
results.append(baseline_metrics)

vit_metrics, _, vit_scores = evaluate_binary_model(
    model_pure_vit,
    test_generator,
    "Pure ViT",
    threshold=0.5,
)
results.append(vit_metrics)

hybrid_metrics, _, hybrid_scores = evaluate_binary_model(
    model_hybrid,
    test_generator,
    "Hybrid ConvNeXt V2 - ViT",
    threshold=0.5,
)
results.append(hybrid_metrics)

import pandas as pd

results_df = pd.DataFrame(results)
display(results_df)

## 12. True ROC Curves and AUC

This section calculates ROC curves from the **actual prediction scores** produced by each model.

This is preferable to manually synthesizing a curve from one confusion matrix operating point.

In [ ]:
plt.figure(figsize=(9, 7))

for model_name, scores in [
    ("ConvNeXt V2 Baseline", baseline_scores),
    ("Pure ViT", vit_scores),
    ("Hybrid ConvNeXt V2 - ViT", hybrid_scores),
]:
    fpr, tpr, _ = roc_curve(
        y_true,
        scores,
    )
    auc_value = roc_auc_score(
        y_true,
        scores,
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{model_name} (AUC={auc_value:.3f})",
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(
    "roc_curve_true.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 13. Training Curves

The following utility plots the actual history objects generated during training.

In [ ]:
def plot_history(history, model_name):
    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 4),
    )

    axes[0].plot(
        history.history["loss"],
        label="Train Loss",
    )
    axes[0].plot(
        history.history["val_loss"],
        label="Val Loss",
    )
    axes[0].set_title(
        f"{model_name} - Loss"
    )
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(
        history.history["accuracy"],
        label="Train Accuracy",
    )
    axes[1].plot(
        history.history["val_accuracy"],
        label="Val Accuracy",
    )
    axes[1].set_title(
        f"{model_name} - Accuracy"
    )
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_history(
    history_baseline,
    "ConvNeXt V2 Baseline",
)

plot_history(
    history_pure_vit,
    "Pure ViT",
)

plot_history(
    history_hybrid,
    "Hybrid ConvNeXt V2 - ViT",
)

## 14. Explainable AI — Grad-CAM Note

The original notebook contains a Grad-CAM attempt based on the layer name `conv5_block3_out`. That layer belongs to a different architecture and does not exist in the custom Hybrid model defined here.

Therefore, the original Grad-CAM cell is **not copied unchanged** into this cleaned notebook. A correct Grad-CAM implementation should first identify an actual spatial convolution layer in the trained custom Hybrid model and then compute gradients with respect to that layer.

This avoids presenting an invalid heatmap as scientific evidence.

In [ ]:
# Inspect convolutional layers in the Hybrid model.
# Use this list to choose an actual spatial layer for a future
# Grad-CAM implementation.

for layer in model_hybrid.layers:
    if isinstance(
        layer,
        (
            layers.Conv2D,
            layers.DepthwiseConv2D,
        ),
    ):
        print(
            layer.name,
            layer.output.shape,
        )

## 15. Optional Gradio Demo

The original project included a Gradio interface. The cleaned version below uses the same **224 × 224** preprocessing size as training.

The interface is intended as a demonstration of model inference, not clinical diagnosis.

In [ ]:
import gradio as gr

def predict_pneumonia(image):
    if image is None:
        return "Please upload a chest X-ray image."

    image = cv2.resize(
        image,
        IMG_SIZE,
    )

    processed = pneumonia_preprocessing(
        image
    )
    processed = processed / 255.0
    processed = np.expand_dims(
        processed,
        axis=0,
    )

    probability = float(
        model_hybrid.predict(
            processed,
            verbose=0,
        )[0][0]
    )

    label = (
        "Pneumonia"
        if probability >= 0.5
        else "Normal"
    )

    return (
        f"Prediction: {label}\n"
        f"Pneumonia probability: "
        f"{probability * 100:.2f}%\n\n"
        "Research/education demo only."
    )


demo = gr.Interface(
    fn=predict_pneumonia,
    inputs=gr.Image(
        type="numpy",
        label="Chest X-Ray",
    ),
    outputs=gr.Textbox(
        label="Result"
    ),
    title="🫁 Pneumonia Detection Demo",
    description=(
        "Hybrid ConvNeXt V2-style + "
        "Multi-Head Attention model with CLAHE."
    ),
)

# Uncomment to launch:
# demo.launch(share=True)

## 16. Project Summary

### Pipeline

```text
Chest X-Ray
    ↓
Resize 224×224
    ↓
Grayscale
    ↓
CLAHE
    ↓
RGB conversion + normalization
    ↓
┌───────────────────────┬───────────────────┬─────────────────────────┐
│ ConvNeXt V2 Baseline  │ Pure ViT          │ Hybrid ConvNeXt V2 +   │
│                       │                   │ Multi-Head Attention    │
└───────────────────────┴───────────────────┴─────────────────────────┘
    ↓
Binary classification
    ↓
Normal / Pneumonia
    ↓
Accuracy / Precision / Recall / F1 / ROC-AUC
```

### Important reproducibility notes

- The cleaned notebook uses the final 224×224 preprocessing pipeline represented in the original notebook.
- The original notebook contained duplicate experiments, inconsistent paths, an incorrect `dataser` path, a 256×256 inference resize, and a Grad-CAM layer name from another architecture. These issues are removed or explicitly documented here.
- Reported numerical results from the original project should be kept in the thesis/report as **experiment results** and should not be silently replaced by newly generated values.
- When the notebook is rerun from scratch, record the new metrics separately because random initialization, runtime, TensorFlow version, and training conditions can affect results.

## 17. Limitations

The dataset used in this project is limited in demographic and clinical diversity. The model should not be interpreted as a clinically validated diagnostic system.

Future work may include:

- External validation on independent datasets
- More rigorous patient-level data splitting
- Better calibration of prediction probabilities
- A validated Grad-CAM implementation for the custom architecture
- Model checkpoint/version management
- Deployment with an appropriate clinical evaluation protocol